## Preparação do Ambiente e Carregamento dos Dados

### Instalação de Pacotes

In [1]:
! pip install --upgrade -q pandas<3.0.0 numpy scikit-learn imbalanced-learn optuna plotly nbformat

/bin/bash: line 1: 3.0.0: No such file or directory


### Importação de Bibliotecas

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline

import optuna
import pickle
import optuna.visualization as vis

#from utils.constants import *

### Constantes

In [3]:
# --- Training/Testing Constants ---

RANDOM_STATE = 42
TEST_RATIO = 0.15
N_FOLDS = 5
N_CLASSES = 3

TARGET_NAMES = ["low_risk", "alarm", "severe"]
TARGET_NAMES_COARSE = ["low_risk", "high_risk"]
TARGET_NAMES_FINE = ["alarm", "severe"]

TARGET_LABEL_MAP = {name: idx for idx, name in enumerate(TARGET_NAMES)}
LABEL_TARGET_MAP = {idx: name for idx, name in enumerate(TARGET_NAMES)}

### Carregamento dos Dados

In [4]:
df = pd.read_csv("../input/sinan-sbcas/dataset-processed-rf.csv")

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

## Treinamento e Otimização do Modelo

### Treinamento com Validação Cruzada

In [ ]:
# def get_pipeline(params, class_counts=None):
#     steps = []

#     if class_counts:
#         n_low, n_alarm, n_severe = class_counts[0], class_counts[1], class_counts[2]
#         steps.append(('under', RandomUnderSampler(sampling_strategy={0: max(n_alarm, n_low // 2)}, random_state=RANDOM_STATE)))
#         steps.append(('over', SMOTENC(categorical_features=categorical_features, sampling_strategy={1: max(n_severe, n_alarm // 2)}, random_state=RANDOM_STATE)))

#     steps.append(('classifier', RandomForestClassifier(**params)))
#     return Pipeline(steps=steps)


def train_on_folds(params):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        # model = get_pipeline(params)

        model = RandomForestClassifier(**params)

        try:
            model.fit(X_train, y_train)
            preds = model.predict(X_valid)
            
            f1 = f1_score(y_valid, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0

    return np.mean(scores), np.std(scores)

### Definição da Função Objetivo

In [5]:
FIXED_PARAMS = {
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    'max_features': 'log2'
}

In [ ]:
def objective(trial: optuna.trial.Trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 250, step=5),
        'max_depth': trial.suggest_int('max_depth', 30, 42),
        'min_samples_split': trial.suggest_int('min_samples_split', 4, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 2),
        **FIXED_PARAMS
    }
    avg, _ = train_on_folds(params)
    
    return avg

### Otimização com Optuna

In [7]:
study = optuna.create_study(direction='maximize')
study.enqueue_trial({'n_estimators': 175, 'max_depth': 33, 'min_samples_split': 6, 'min_samples_leaf': 2})
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

[I 2026-02-08 05:16:51,574] A new study created in memory with name: no-name-c93b4204-5d01-4f78-a7e0-2e78ea62577b


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-08 05:34:14,481] Trial 0 finished with value: 0.5708022145334282 and parameters: {'n_estimators': 175, 'max_depth': 33, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.5708022145334282.
[I 2026-02-08 05:36:55,344] Trial 2 finished with value: 0.57346192874468 and parameters: {'n_estimators': 200, 'max_depth': 34, 'min_samples_split': 4, 'min_samples_leaf': 2}. Best is trial 2 with value: 0.57346192874468.
[I 2026-02-08 05:37:15,671] Trial 3 finished with value: 0.5708602366083462 and parameters: {'n_estimators': 205, 'max_depth': 41, 'min_samples_split': 7, 'min_samples_leaf': 2}. Best is trial 2 with value: 0.57346192874468.
[I 2026-02-08 05:38:04,347] Trial 1 finished with value: 0.5714443111409924 and parameters: {'n_estimators': 215, 'max_depth': 36, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 2 with value: 0.57346192874468.
[I 2026-02-08 05:55:38,163] Trial 7 finished with value: 0.5737208323993307 and parameters: {'n_esti

In [ ]:
output_dir = Path('opt_rf')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

In [ ]:
display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

### Melhor Modelo

In [11]:
final_params = {**best_trial.params, **FIXED_PARAMS}

In [12]:
avg_f1_final, std_f1_final = train_on_folds(final_params)

In [13]:
print(avg_f1_final)
print(std_f1_final)

0.5738859974082489
0.004739264831124382
